# California housing price prediction

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/camnugent/california-housing-prices/housing.csv')
df.head()

## Basic checks

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## Cleaning

In [ ]:
# Fill missing bedrooms with the median
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

# Remove duplicate rows if present
df = df.drop_duplicates().reset_index(drop=True)

df.isnull().sum()

## Prepare the data

In [ ]:
# Convert ocean_proximity into numeric columns
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)
df.head()

In [ ]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Linear Regression

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

linear_pred = linear_model.predict(X_test_scaled)

## Ridge

In [ ]:
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)

ridge_pred = ridge_model.predict(X_test_scaled)

## Lasso

In [ ]:
lasso_model = Lasso(alpha=100.0, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)

lasso_pred = lasso_model.predict(X_test_scaled)

## Results

In [ ]:
linear_mae = mean_absolute_error(y_test, linear_pred)
linear_mse = mean_squared_error(y_test, linear_pred)
linear_rmse = np.sqrt(linear_mse)
linear_r2 = r2_score(y_test, linear_pred)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_mse = mean_squared_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(ridge_mse)
ridge_r2 = r2_score(y_test, ridge_pred)

lasso_mae = mean_absolute_error(y_test, lasso_pred)
lasso_mse = mean_squared_error(y_test, lasso_pred)
lasso_rmse = np.sqrt(lasso_mse)
lasso_r2 = r2_score(y_test, lasso_pred)

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'Lasso'],
    'MAE': [linear_mae, ridge_mae, lasso_mae],
    'MSE': [linear_mse, ridge_mse, lasso_mse],
    'RMSE': [linear_rmse, ridge_rmse, lasso_rmse],
    'R2': [linear_r2, ridge_r2, lasso_r2]
})

results.round(3)

## Coefficients

In [ ]:
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Linear': linear_model.coef_,
    'Ridge': ridge_model.coef_,
    'Lasso': lasso_model.coef_
})

coefficients.round(3)

In [ ]:
# Number of coefficients reduced to zero by Lasso
(coefficients['Lasso'] == 0).sum()

## Conclusion

In [ ]:
best_model = results.loc[results['RMSE'].idxmin()]

print(f"{best_model['Model']} generalizes best because it has the lowest test RMSE ({best_model['RMSE']:.2f}) and an R² of {best_model['R2']:.3f}.")
print("Ridge shrinks the coefficients slightly, which can reduce sensitivity to correlated features without removing them.")
print("Lasso applies stronger shrinkage and can set less useful coefficients to zero, but too much regularization may slightly reduce test performance.")